# 从经典 Hopfield 到现代 Hopfield：MNIST 联想记忆实验

**[目标]**：用同一批 MNIST 图像，比较经典 Hopfield、二值 Dense Associative Memory、连续 Modern Hopfield 如何从下半部分遮挡的线索恢复原图。

**[来源]**：改写自 [Hopfield Networks is all you Need](https://ml-jku.github.io/hopfield-layers/) 的教学 notebook。

**[重构原则]**：删除只展示输入、不产生检索或测量结果的代码；每个实验必须明确输入、存储、线索、更新、测量和展示。


## 0. 实验管线

|阶段|组件|回答的问题|
|---|---|---|
|a. 输入编码|`a1_load_mnist`、`a2_binarize_patterns`|网络收到什么状态？|
|b. 存储|`b1_store_classical`、`b2_store_binary_dense`、`b3_store_continuous`|记忆以权重矩阵还是图样矩阵保存？|
|c. 检索线索|`c1_make_binary_cue`、`c2_make_continuous_cue`|如何制造缺失下半部分的输入？|
|d. 动力学检索|`d1_run_classical`、`d2_binary_dense_update`、`d3_continuous_update`|网络如何从线索移动到输出？|
|e. 测量|`e1_binary_bit_error`、`e2_continuous_mse`|输出离目标有多远？|
|f. 展示|`f1_plot_metric`、`f2_show_retrieval`|如何比较条件并检查代表样本？|

**[关键区别]**：经典网络把记忆压入一个 `784×784` 权重矩阵；两种现代网络直接保留记忆图样矩阵。


## 1. a：输入与编码

MNIST 像素原本位于 `[0,1]`。经典网络与二值 Dense Associative Memory 使用 `{-1,+1}`；连续 Modern Hopfield 保留灰度值。

**[修正]**：编码函数只负责“把图像变成状态”。动力学的符号判决由 `d0_sign` 单独完成，避免把图像编码规则误当成 Hopfield 更新规则。


In [ ]:
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torchvision import transforms

SEED = 0
torch.manual_seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

# [环境] Colab 默认可能缺少中文字形；下载失败时仍可运行，只是标题可能缺字。
font_path = Path("NotoSansCJKtc-Regular.otf")
if not font_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(
            "https://github.com/notofonts/noto-cjk/raw/main/"
            "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf",
            font_path,
        )
    except Exception:
        pass
if font_path.exists():
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False


def a1_load_mnist(
    batch_size: int = 1000,
    seed: int = SEED,
) -> tuple[torch.Tensor, torch.Tensor]:
    """[输入] 下载 MNIST；[输出] images:(B,784)、labels:(B,)。"""
    dataset = torchvision.datasets.MNIST(
        root="./mnist_data",
        train=True,
        download=True,
        transform=transforms.ToTensor(),
    )
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )
    images, labels = next(iter(loader))
    return images.reshape(images.shape[0], 784).float(), labels


def a2_binarize_patterns(images: torch.Tensor) -> torch.Tensor:
    """[编码] [0,1] 灰度图 → {-1,+1} 二值图样。"""
    return torch.where(images > 0, -torch.ones_like(images), torch.ones_like(images))


images, labels = a1_load_mnist(batch_size=1000)
binary_images = a2_binarize_patterns(images)

print("图像矩阵 (images)：", tuple(images.shape))
print("标签向量 (labels)：", tuple(labels.shape))


## 2. b：存储

**[经典]**：`W = X.T @ X`，多条记忆叠加到同一个权重矩阵；主对角线清零，去掉自连接。

**[二值 Dense Associative Memory]**：直接保存二值图样矩阵；检索时重新比较线索与每条记忆。

**[连续 Modern Hopfield]**：直接保存连续图样矩阵；检索时用 softmax 产生记忆权重。


In [ ]:
def b1_store_classical(patterns: torch.Tensor) -> torch.Tensor:
    """[存储] patterns:(n,D) → 对称权重 weights:(D,D)。"""
    weights = patterns.T @ patterns
    weights.fill_diagonal_(0.0)
    return weights


def b2_store_binary_dense(patterns: torch.Tensor) -> torch.Tensor:
    """[存储] 二值 Dense Associative Memory 直接保留图样。"""
    return patterns.clone()


def b3_store_continuous(patterns: torch.Tensor) -> torch.Tensor:
    """[存储] 连续 Modern Hopfield 直接保留灰度图样。"""
    return patterns.clone()


## 3. c：构造检索线索

遮挡不是另一个数据集，而是从目标记忆制造的部分线索。

- 二值模型：下半部分设为 `-1`。
- 连续模型：下半部分设为 `0`。


In [ ]:
def _reshape_image(pattern: torch.Tensor) -> torch.Tensor:
    """[约束] 本实验只处理 28×28 MNIST 图像。"""
    if pattern.numel() != 784:
        raise ValueError("pattern 必须包含 784 个像素")
    return pattern.reshape(28, 28)


def c1_make_binary_cue(pattern: torch.Tensor) -> torch.Tensor:
    """[线索] 遮挡二值图样的下半部分。"""
    cue = _reshape_image(pattern).clone()
    cue[14:, :] = -1
    return cue.flatten()


def c2_make_continuous_cue(pattern: torch.Tensor) -> torch.Tensor:
    """[线索] 遮挡连续图样的下半部分。"""
    cue = _reshape_image(pattern).clone()
    cue[14:, :] = 0
    return cue.flatten()


## 4. d：动力学检索

### d1 经典 Hopfield

反复计算局部场 `h = W @ state`，再用符号函数更新，直到状态不再改变或达到步数上限。

### d2 二值 Dense Associative Memory

对每个待更新比特，分别计算它取 `+1` 和 `-1` 时的指数匹配分数。代码使用 `logsumexp`：与比较指数和等价，但数值更稳定。

### d3 连续 Modern Hopfield

`attention = softmax(β · X @ cue)`，再用 `X.T @ attention` 读出记忆。


In [ ]:
def d0_sign(field: torch.Tensor, previous: torch.Tensor) -> torch.Tensor:
    """[判断] 正场取 +1，负场取 -1，零场保持原状态。"""
    return torch.where(
        field > 0,
        torch.ones_like(field),
        torch.where(field < 0, -torch.ones_like(field), previous),
    )


def d1_run_classical(
    weights: torch.Tensor,
    cue: torch.Tensor,
    max_steps: int = 20,
) -> tuple[torch.Tensor, int]:
    """[更新] 同步迭代到固定点或 max_steps。"""
    state = cue.clone()
    for step in range(1, max_steps + 1):
        new_state = d0_sign(weights @ state, state)
        if torch.equal(new_state, state):
            return state, step
        state = new_state
    return state, max_steps


def d2_binary_dense_update(
    memories: torch.Tensor,
    cue: torch.Tensor,
    temperature: float = 10.0,
) -> torch.Tensor:
    """[更新] 向量化比较每个比特取 ±1 时的 Dense Memory 分数。"""
    z = cue.flatten()
    base_overlap = memories @ z

    positive_overlap = (
        base_overlap[:, None]
        + memories * (1.0 - z)[None, :]
    )
    negative_overlap = (
        base_overlap[:, None]
        + memories * (-1.0 - z)[None, :]
    )

    positive_score = torch.logsumexp(positive_overlap / temperature, dim=0)
    negative_score = torch.logsumexp(negative_overlap / temperature, dim=0)
    return torch.where(
        positive_score > negative_score,
        torch.ones_like(positive_score),
        -torch.ones_like(negative_score),
    )


def d3_continuous_update(
    memories: torch.Tensor,
    cue: torch.Tensor,
    beta: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """[更新] softmax 加权读取连续记忆。"""
    similarities = memories @ cue
    attention = F.softmax(beta * similarities, dim=0)
    output = memories.T @ attention
    return output, attention


## 5. e / f：测量与展示

**[二值误差]**：输出与目标不同的像素比例。  
**[连续误差]**：输出与目标之间的均方误差（MSE）。  
**[展示边界]**：只展示每组实验的一个代表样本；批量结果用误差曲线汇总。


In [ ]:
def e1_binary_bit_error(
    output: torch.Tensor,
    target: torch.Tensor,
) -> float:
    """[测量] 二值输出的错误像素比例。"""
    return float(torch.mean((output != target).float()))


def e2_continuous_mse(
    output: torch.Tensor,
    target: torch.Tensor,
) -> float:
    """[测量] 连续输出的均方误差。"""
    return float(F.mse_loss(output, target))


def f1_plot_metric(
    x,
    values,
    xlabel: str,
    ylabel: str,
    title: str,
) -> None:
    """[展示] 汇总不同实验条件的量化结果。"""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, values, marker="o")
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    plt.show()


def f2_show_retrieval(
    target: torch.Tensor,
    cue: torch.Tensor,
    output: torch.Tensor,
    title: str,
) -> None:
    """[展示] 一个代表样本：目标、线索、检索输出。"""
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    panels = [
        ("目标图样", target),
        ("遮挡线索", cue),
        ("检索输出", output),
    ]
    for ax, (panel_title, image) in zip(axes, panels):
        ax.imshow(image.reshape(28, 28), cmap="gray")
        ax.set_title(panel_title)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 6. 公共组件自检

先验证组件边界，再运行 MNIST 实验。自检不展示图片。


In [ ]:
toy_memories = torch.tensor(
    [[1.0, 1.0, -1.0, -1.0], [-1.0, 1.0, -1.0, 1.0]]
)
toy_weights = b1_store_classical(toy_memories)

assert torch.allclose(toy_weights, toy_weights.T)
assert torch.allclose(torch.diag(toy_weights), torch.zeros(4))

toy_dense = d2_binary_dense_update(toy_memories, toy_memories[0])
assert set(torch.unique(toy_dense).tolist()) <= {-1.0, 1.0}

toy_output, toy_attention = d3_continuous_update(
    toy_memories,
    toy_memories[0],
    beta=1.0,
)
assert toy_output.shape == toy_memories[0].shape
assert torch.allclose(toy_attention.sum(), torch.tensor(1.0))

print("公共组件自检通过")


## 7. 实验 1：经典 Hopfield 的相关记忆干扰

**[问题]**：存储 MNIST 图样数从 1 增加到 3、10 时，二值检索误差如何变化？

**[配方]**：`a2 → b1 → c1 → d1 → e1 → f1/f2`

每个条件最多评估前 10 条记忆；误差曲线是主要结果，三联图只展示 `n=10` 的一个代表样本。


In [ ]:
classical_n_values = [1, 3, 10]
classical_errors = []
classical_example = None

for n in classical_n_values:
    memories = binary_images[:n]
    weights = b1_store_classical(memories)

    errors = []
    for target_index in range(min(n, 10)):
        target = memories[target_index]
        cue = c1_make_binary_cue(target)
        output, steps = d1_run_classical(weights, cue)
        errors.append(e1_binary_bit_error(output, target))

        if n == 10 and target_index == 0:
            classical_example = (target, cue, output)

    mean_error = sum(errors) / len(errors)
    classical_errors.append(mean_error)
    print(f"经典 Hopfield：n={n}，平均错误率={mean_error:.3f}")

f1_plot_metric(
    classical_n_values,
    classical_errors,
    xlabel="存储图样数 n",
    ylabel="平均错误像素比例",
    title="经典 Hopfield：相关图样增加时的检索误差",
)
f2_show_retrieval(
    *classical_example,
    title="经典 Hopfield 代表样本（n=10）",
)


**[解释]**：这里观察到的是“相关 MNIST 图样造成的交叉干扰”，不能直接当作随机独立图样的理论容量。


## 8. 实验 2：二值 Dense Associative Memory

**[问题]**：把存储机制换成指数匹配后，`n=10` 和 `n=100` 的检索误差如何变化？

**[配方]**：`a2 → b2 → c1 → d2 → e1 → f1/f2`

两个条件都评估前 10 条记忆，保持测量口径一致。


In [ ]:
dense_n_values = [10, 100]
dense_errors = []
dense_example = None

for n in dense_n_values:
    memories = b2_store_binary_dense(binary_images[:n])

    errors = []
    for target_index in range(10):
        target = memories[target_index]
        cue = c1_make_binary_cue(target)
        output = d2_binary_dense_update(memories, cue)
        errors.append(e1_binary_bit_error(output, target))

        if n == 100 and target_index == 0:
            dense_example = (target, cue, output)

    mean_error = sum(errors) / len(errors)
    dense_errors.append(mean_error)
    print(f"二值 Dense Memory：n={n}，平均错误率={mean_error:.3f}")

f1_plot_metric(
    dense_n_values,
    dense_errors,
    xlabel="存储图样数 n",
    ylabel="平均错误像素比例",
    title="二值 Dense Associative Memory 检索误差",
)
f2_show_retrieval(
    *dense_example,
    title="二值 Dense Memory 代表样本（n=100）",
)


**[解释]**：高阶匹配提高了相似图样之间的区分力；但“理论容量很大”仍不意味着高度相关的 MNIST 图样一定全部可分。


## 9. 实验 3：连续 Modern Hopfield

**[问题]**：连续检索在 `n=10、100、1000` 时的 MSE 如何变化？

**[配方]**：`a1 → b3 → c2 → d3 → e2 → f1/f2`

每个条件评估前 20 条记忆，默认 `β=8`。


In [ ]:
continuous_n_values = [10, 100, 1000]
continuous_errors = []
continuous_example = None
beta = 8.0

for n in continuous_n_values:
    memories = b3_store_continuous(images[:n])

    errors = []
    for target_index in range(min(n, 20)):
        target = memories[target_index]
        cue = c2_make_continuous_cue(target)
        output, attention = d3_continuous_update(memories, cue, beta=beta)
        errors.append(e2_continuous_mse(output, target))

        if n == 1000 and target_index == 0:
            continuous_example = (target, cue, output)

    mean_error = sum(errors) / len(errors)
    continuous_errors.append(mean_error)
    print(f"连续 Modern Hopfield：n={n}，平均 MSE={mean_error:.6f}")

f1_plot_metric(
    continuous_n_values,
    continuous_errors,
    xlabel="存储图样数 n",
    ylabel="平均 MSE",
    title="连续 Modern Hopfield 检索误差",
)
f2_show_retrieval(
    *continuous_example,
    title="连续 Modern Hopfield 代表样本（n=1000）",
)


## 10. 实验 4：β 控制什么？

**[问题]**：固定 100 条记忆时，β 如何改变 softmax 的选择锐度和检索误差？

**[自变量]**：`β = 0.1、0.2、0.5、1、2、4、8`。  
**[因变量]**：同一条遮挡线索的 MSE。


In [ ]:
beta_values = [0.1, 0.2, 0.5, 1, 2, 4, 8]
beta_memories = b3_store_continuous(images[:100])
beta_target = beta_memories[0]
beta_cue = c2_make_continuous_cue(beta_target)
beta_errors = []

for beta in beta_values:
    output, attention = d3_continuous_update(
        beta_memories,
        beta_cue,
        beta=beta,
    )
    error = e2_continuous_mse(output, beta_target)
    beta_errors.append(error)
    print(
        f"β={beta:>3}，MSE={error:.6f}，"
        f"最大注意力权重={float(attention.max()):.4f}"
    )

f1_plot_metric(
    beta_values,
    beta_errors,
    xlabel="β",
    ylabel="MSE",
    title="β 对连续 Modern Hopfield 检索的影响",
)


## 11. 如何比较三种网络？

- **经典 Hopfield**：记忆压缩进权重矩阵；相关记忆产生交叉干扰。
- **二值 Dense Associative Memory**：直接比较线索与所有二值记忆；匹配更尖锐，但仍受相关性影响。
- **连续 Modern Hopfield**：softmax 权重直接选择连续记忆；β 控制选择锐度。

三组误差的定义不同：二值模型使用错误像素比例，连续模型使用 MSE。因此不要直接比较柱高；应分别观察每种模型随存储数增加的趋势。
